# VLM LoRA Ablation Study — Full Pipeline

**Before running:**
- Notebook settings → Accelerator → **GPU T4 x1**
- Notebook settings → Internet → **On**
- Upload the repo zip as a Kaggle Dataset and attach it to this notebook as input.

Run cells **top to bottom in order**. Cell 7 (smoke test) is a hard gate — if it raises an error,
**stop and debug before continuing** (see HANDOFF.md §5).

Expected total wall-clock time on a free T4: **~6–10 hours** (dominated by the ablation matrix).

## Cell 1 — Repo setup + exact pinned dependencies

These pins were arrived at after ~15 rounds of trial-and-error (HANDOFF.md §3). **Do not change them without a reason.**

- `transformers==4.46.3` / `tokenizers==0.20.3`: Kaggle default (5.0.0) crashes against Moondream2's custom modeling class.
- tensorflow/scipy removed: broken numpy/scipy pairing on this Kaggle image surfaces through that import chain.
- `datasets==2.20.0`: newer datasets dropped the legacy loading-script format VQAv2 still uses.

In [ ]:
import os, subprocess, sys

# ── 1a. Locate and copy repo from attached Kaggle Dataset input ───────────────
# Replace with the slug shown under /kaggle/input/ after attaching your dataset
DATASET_SLUG = "vlm-lora-study"   # <- change this if your dataset slug differs

src = f"/kaggle/input/{DATASET_SLUG}/vlm-lora-study"

if not os.path.exists(src):
    for root, dirs, _ in os.walk('/kaggle/input'):
        if "vlm-lora-study" in dirs:
            src = os.path.join(root, "vlm-lora-study")
            break
    else:
        raise FileNotFoundError(
            "Could not find vlm-lora-study under /kaggle/input. "
            "Make sure the dataset is attached and update DATASET_SLUG above."
        )

print(f"Copying repo from {src} -> /kaggle/working/vlm-lora-study")
subprocess.run(["cp", "-r", src, "/kaggle/working/"], check=True)
os.chdir("/kaggle/working/vlm-lora-study")
print("Working directory:", os.getcwd())
print("Repo contents:", os.listdir("."))

# ── 1b. Install pinned dependencies (order matters — HANDOFF §3) ─────────────
!pip install -q --force-reinstall "transformers==4.46.3" "tokenizers==0.20.3"
!pip uninstall -y -q tensorflow scipy
!pip install -q "datasets==2.20.0"
!pip install -q peft pillow pyyaml pandas matplotlib fastapi "uvicorn[standard]" python-multipart einops wandb tqdm bitsandbytes

print("\n[OK] Installation complete.")

## Cell 2 — Verify the transformers pin took effect

Kaggle's kernel may pre-import transformers before your pip install runs, silently defeating the pin.
This cell hot-reloads it from sys.modules if needed.

In [ ]:
import sys

# Flush any stale cached version from sys.modules
for m in list(sys.modules):
    if m == "transformers" or m.startswith("transformers."):
        del sys.modules[m]

import transformers
ver = transformers.__version__
print(f"transformers version: {ver}")

if ver != "4.46.3":
    raise RuntimeError(
        f"transformers is {ver}, expected 4.46.3. "
        "Restart the kernel (WITHOUT Run All), then re-run from Cell 1. "
        "See HANDOFF.md §3 for more detail."
    )

print("[OK] Correct transformers version confirmed.")

## Cell 3 — Sanity-load Moondream2

Downloads ~4 GB of weights on first run (Kaggle caches them after that).
Confirms the base model loads cleanly with the pinned transformers.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading Moondream2 in fp16...")
_model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    torch_dtype=torch.float16,  # torch_dtype= not dtype= in this transformers version
).to("cuda")
_tok = AutoTokenizer.from_pretrained("vikhyatk/moondream2")

print("\n[OK] Model loaded OK")
print(f"Peak VRAM after load: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

# Free the test load — pipeline scripts load the model fresh via load_moondream()
del _model, _tok
torch.cuda.empty_cache()
print("Temp model freed. Ready to run pipeline.")

## Cell 4 — inspect_model.py (diagnostic — read-only)

Dumps the live module tree and source of the key forward-pass functions `train.py` depends on.
Compare against the assumptions in `scripts/common.py` docstrings — see HANDOFF.md §5 for what to look for.

This cell does not modify anything.

In [ ]:
!python scripts/inspect_model.py

## Cell 5 — prepare_data.py

Streams ~3300 VQAv2 examples from the Hub, saves images to `data/raw/images/`, and writes:
- `data/test.jsonl` — 300 held-out examples
- `data/train_unfiltered.jsonl` — ~3000 examples (all answers)
- `data/train_filtered.jsonl` — non-trivial answers only (yes/no/digits removed)

**Expected wall-clock: ~50 minutes.** Skipped automatically if the three files already exist.

In [ ]:
import os

files = ["data/test.jsonl", "data/train_unfiltered.jsonl", "data/train_filtered.jsonl"]

if all(os.path.exists(p) and os.path.getsize(p) > 0 for p in files):
    print("Data files already exist — skipping prepare_data.py")
    for p in files:
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f"  {p}: {n} lines")
else:
    print("Running prepare_data.py (~50 minutes over the network)...")
    !python scripts/prepare_data.py

for p in files:
    assert os.path.exists(p) and os.path.getsize(p) > 0, f'Missing or empty: {p}'
print("\n[OK] Data files verified.")

## Cell 6 — Zero-shot baseline

Runs Moondream2 (no adapter) on the 300-example test set. Records exact-match accuracy in
`results/metrics.csv` as the `zero_shot_baseline` row.
Every fine-tuned config's delta is computed against this number.

**Expected wall-clock: ~15–20 minutes.** Skipped if already recorded in metrics.csv.

In [ ]:
import os, pandas as pd

metrics_csv = "results/metrics.csv"

def baseline_already_done():
    if not os.path.exists(metrics_csv):
        return False
    df = pd.read_csv(metrics_csv)
    return "zero_shot_baseline" in df.get("config_name", pd.Series()).values

if baseline_already_done():
    df = pd.read_csv(metrics_csv)
    row = df[df["config_name"] == "zero_shot_baseline"].iloc[-1]
    print("Zero-shot baseline already recorded — skipping.")
    print(f"  accuracy: {row['accuracy']}")
else:
    print("Running zero-shot baseline (~15–20 minutes)...")
    !python scripts/baseline.py

df = pd.read_csv(metrics_csv)
baseline_acc = float(df[df["config_name"] == "zero_shot_baseline"].iloc[-1]["accuracy"])
print(f"\n[OK] Zero-shot baseline accuracy: {baseline_acc:.4f}")

## Cell 7 — Smoke test ⚠️ HARD GATE — do not skip

Runs **one** forward + backward pass on a single example and confirms gradients reach the LoRA parameters.

> **If this cell raises an error: STOP. Do NOT run Cell 8 (ablation).**
> See HANDOFF.md §5 — re-read Cell 4's output side-by-side with `_attn_no_cache_with_lora` in `scripts/common.py`.
> The bug is almost certainly a mismatch there, not somewhere else.

Expected success output ends with: `Smoke test passed.`

In [ ]:
import subprocess, sys

print("Running smoke test (one forward+backward, ~30 seconds)...")
result = subprocess.run(
    [sys.executable, 'scripts/train.py', '--smoke_test'],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        '\n' + '='*70 + '\n'
        'SMOKE TEST FAILED (non-zero exit code).\n'
        'Do NOT proceed to Cell 8 (ablation.py).\n'
        'See HANDOFF.md section 5 for the debugging procedure.\n'
        + '='*70
    )

print("\n[OK] Smoke test passed — safe to run the full ablation.")

## Cell 8 — Full ablation matrix

`ablation.py` orchestrates the full sweep using subprocesses (one per run, for GPU memory isolation):

| Step | Sweep | New runs |
|------|-------|----------|
| 1 | Rank ∈ {4,8,16,32} × cosine × filtered | 4 |
| 2 | Best rank × {cosine, constant} × filtered | 1 (cosine already done) |
| 3 | Best rank × cosine × {filtered, unfiltered} | 1 (filtered already done) |

Each run: `train.py` (fine-tune) → `evaluate.py` (eval) → row appended to `results/metrics.csv`.

**Expected wall-clock: 5–8 hours total on a free T4.**
Partial progress is preserved — already-completed runs are skipped automatically on re-run.

In [ ]:
!python scripts/ablation.py

## Cell 9 — Plots + final results table

Generates:
- `results/accuracy_vs_rank.png` — accuracy vs. LoRA rank (cosine, filtered data)
- `results/filtered_vs_unfiltered.png` — data filtering effect bar chart

Then prints the full metrics table for copying into the README.

In [ ]:
import os, pandas as pd
from IPython.display import Image as IPImage, display

!python results/plots.py

print('\n' + '='*88)
print('FULL METRICS TABLE')
print('='*88)
df = pd.read_csv('results/metrics.csv')
print(df.to_string(index=False))

print('\n' + '='*88)
print('PLOTS')
print('='*88)
for img in ['results/accuracy_vs_rank.png', 'results/filtered_vs_unfiltered.png']:
    if os.path.exists(img):
        display(IPImage(img))
    else:
        print(f'(not generated: {img})')

## Next steps after this notebook completes

1. **Download outputs**: Save `results/metrics.csv` + both PNGs from the Kaggle output panel.
2. **Fill in the README**: Replace the `*To be written*` placeholder tables with real numbers from `metrics.csv`.
3. **Update HANDOFF.md §2**: Mark steps as confirmed working.
4. **Optional — serve the best adapter**:
   ```bash
   # Edit config.yaml: set serve.adapter_path to the best models/rank* dir, then:
   python scripts/serve.py
   ```
   See `scripts/serve.py` for the FastAPI endpoint details.